[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yryo1005/OpenCampus_Demo/blob/main/OC_FacialExpression.ipynb)


# 表情認識デモ（Facial Expression Recognition）

カメラや写真から顔を検出し，**7種類の表情**（喜び・悲しみ・怒り・驚き・恐れ・嫌悪・無表情）を推定するデモです．  
Hugging Face の Vision Transformer（[trpakov/vit-face-expression](https://huggingface.co/trpakov/vit-face-expression)）を Colab の GPU 上で動かします．

**実行環境**: Google Colab（ランタイム → GPU: T4 推奨）

## セルの進め方
1. **設定**（Webカメラの左右反転など）
2. **ライブラリのインストール**
3. **ライブラリの読み込み・モデル準備・サンプル画像のダウンロード**
4. **Gradio の起動**

> API キーは **不要** です（推論はすべて Colab 内で完結）．  
> Hugging Face のユーザー認証も **不要** です．  
> インストール直後にエラーが出る場合は，**ランタイム → セッションを再起動**してから設定セルとセル3以降を再実行してください．


## 0. 設定

- カメラ映像が左右反転して見える場合は，次のセルの `MIRROR_WEBCAM` を切り替えてください（`True` = ミラー，`False` = 反転なし）．
- 変更後は **Gradio 起動セル**を再実行してください．


In [ ]:
# Webカメラの左右反転（ミラー表示）
# True  : 左右反転する（Gradio のデフォルトに近い自撮り表示）
# False : 左右反転しない
MIRROR_WEBCAM = True

# 表情認識モデル（Hugging Face）
MODEL_ID = "trpakov/vit-face-expression"

# 顔検出後に周囲へ余白を足す倍率（1.0 = 検出枠そのもの）
FACE_MARGIN = 0.25

print(f"MIRROR_WEBCAM = {MIRROR_WEBCAM}")
print(f"MODEL_ID = {MODEL_ID}")
print(f"FACE_MARGIN = {FACE_MARGIN}")


## 1. ライブラリのインストール


In [ ]:
# Colab 標準の torch / transformers / gradio / opencv / Pillow を利用
# 表情認識用に transformers を念のため更新
!pip install -q -U "transformers>=4.40.0"


## 2. ライブラリの読み込み，変数のインスタンス化

サンプル顔写真（日本人・アジア系を含む）をインターネットからダウンロードし，顔検出と表情認識モデルを準備します．  
初回はモデルのダウンロードに数分かかることがあります．


In [ ]:
from __future__ import annotations

import urllib.request
from pathlib import Path

import cv2
import gradio as gr
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm
from transformers import pipeline

# ------------------------------------------------------------
# 定数・サンプル画像 URL
# ------------------------------------------------------------
SAMPLE_DIR = Path("samples_facial_expression")
FONT_DIR = Path("fonts")
FONT_PATH = FONT_DIR / "NotoSansJP-VF.ttf"
FONT_URL = (
    "https://raw.githubusercontent.com/googlefonts/noto-cjk/main/"
    "Sans/Variable/TTF/Subset/NotoSansJP-VF.ttf"
)

# 公開画像（Wikimedia / Pexels / Unsplash）．日本人・アジア系を含む．
# 大きい Wikimedia 画像は DL 後にリサイズする．
SAMPLE_IMAGE_SOURCES: list[tuple[str, str, str]] = [
    (
        "happy_japanese_woman.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/b/b0/Smiling_Japanese_Woman.jpg",
        "喜び（日本人・笑顔）",
    ),
    (
        "happy_japanese_smile.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/d/d1/Smiling_Ai_Hongo_%282024%2902.jpg",
        "喜び（日本人・スマイル）",
    ),
    (
        "neutral_japanese.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/9/90/Geisha_face_%285025641801%29.jpg",
        "無表情寄り（日本人）",
    ),
    (
        "asian_portrait.jpg",
        "https://images.pexels.com/photos/1239291/pexels-photo-1239291.jpeg?auto=compress&cs=tinysrgb&w=640",
        "ポートレート（アジア系）",
    ),
    (
        "laughing.jpg",
        "https://images.pexels.com/photos/1898555/pexels-photo-1898555.jpeg?auto=compress&cs=tinysrgb&w=640",
        "笑い",
    ),
    (
        "serious.jpg",
        "https://images.pexels.com/photos/2379004/pexels-photo-2379004.jpeg?auto=compress&cs=tinysrgb&w=640",
        "真剣な表情",
    ),
    (
        "surprised.jpg",
        "https://images.pexels.com/photos/3755761/pexels-photo-3755761.jpeg?auto=compress&cs=tinysrgb&w=640",
        "驚き寄り",
    ),
]

# 英語ラベル → 日本語表示
EMOTION_JA: dict[str, str] = {
    "angry": "怒り",
    "disgust": "嫌悪",
    "fear": "恐れ",
    "happy": "喜び",
    "sad": "悲しみ",
    "surprise": "驚き",
    "neutral": "無表情",
}

USER_AGENT = "Mozilla/5.0 (compatible; OpenCampusDemo/1.0; +https://github.com/yryo1005/OpenCampus_Demo)"


def resolve_device() -> str:
    """利用可能な推論デバイスを返す．

    Returns:
        str: "cuda" または "cpu"
    """
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"GPU: {name} ({mem_gb:.1f} GB)")
        return "cuda"
    print("GPU が見つかりません．CPU で実行します（時間がかかります）．")
    return "cpu"


def download_file(url: str, save_path: Path, max_side: int = 1280) -> Path:
    """URL から画像をダウンロードし，必要なら長辺を縮小して保存する．

    Args:
        url (str): ダウンロード元 URL
        save_path (Path): 保存先パス
        max_side (int): 長辺の上限ピクセル（既定 1280）

    Returns:
        Path: 保存したファイルのパス
    """
    if save_path.exists() and save_path.stat().st_size > 0:
        return save_path
    save_path.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(req, timeout=60) as response:
        raw = response.read()
    arr = np.frombuffer(raw, dtype=np.uint8)
    bgr = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if bgr is None:
        raise RuntimeError(f"画像のデコードに失敗しました: {url}")
    h, w = bgr.shape[:2]
    long_side = max(h, w)
    if long_side > max_side:
        scale = max_side / float(long_side)
        bgr = cv2.resize(
            bgr,
            (int(w * scale), int(h * scale)),
            interpolation=cv2.INTER_AREA,
        )
    ok, encoded = cv2.imencode(".jpg", bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
    if not ok:
        raise RuntimeError(f"画像のエンコードに失敗しました: {save_path}")
    save_path.write_bytes(encoded.tobytes())
    return save_path


def download_font(url: str, save_path: Path) -> Path:
    """日本語表示用フォントをダウンロードする（既存ならスキップ）．

    Args:
        url (str): フォントの URL
        save_path (Path): 保存先パス

    Returns:
        Path: 保存したフォントのパス
    """
    if save_path.exists() and save_path.stat().st_size > 0:
        return save_path
    save_path.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(req, timeout=120) as response:
        save_path.write_bytes(response.read())
    return save_path


def prepare_sample_images(
    sources: list[tuple[str, str, str]],
    sample_dir: Path,
) -> list[tuple[str, Path]]:
    """サンプル顔写真をダウンロードし，ラベルとパスの一覧を返す．

    Args:
        sources (list[tuple[str, str, str]]): (ファイル名, URL, 表示ラベル) のリスト
        sample_dir (Path): 保存先ディレクトリ

    Returns:
        list[tuple[str, Path]]: (表示ラベル, ローカルパス) のリスト
    """
    prepared: list[tuple[str, Path]] = []
    for filename, url, label in tqdm(sources, desc="サンプル画像DL", leave=False):
        path = download_file(url, sample_dir / filename)
        print(f"  {label}: {path} ({path.stat().st_size} bytes)")
        prepared.append((label, path))
    return prepared


def load_face_cascade() -> cv2.CascadeClassifier:
    """OpenCV の顔検出用 Haar Cascade を読み込む．

    Returns:
        cv2.CascadeClassifier: 顔検出器

    Raises:
        RuntimeError: cascade ファイルが読めない場合
    """
    cascade_path = Path(cv2.data.haarcascades) / "haarcascade_frontalface_default.xml"
    cascade = cv2.CascadeClassifier(str(cascade_path))
    if cascade.empty():
        raise RuntimeError(f"顔検出 cascade を読めません: {cascade_path}")
    return cascade


def load_emotion_pipeline(model_id: str, device: str):
    """表情認識（画像分類）パイプラインを構築する．

    Args:
        model_id (str): Hugging Face モデル ID
        device (str): "cuda" または "cpu"

    Returns:
        transformers.pipelines.Pipeline: image-classification パイプライン
    """
    device_index = 0 if device == "cuda" else -1
    print(f"モデルを読み込み中: {model_id} (device={device})")
    clf = pipeline(
        task="image-classification",
        model=model_id,
        device=device_index,
    )
    return clf


def to_rgb_uint8(image) -> np.ndarray | None:
    """Gradio / PIL / ndarray 入力を RGB uint8 (H, W, 3) に揃える．

    Args:
        image: Gradio Image の入力（None / PIL.Image / np.ndarray）

    Returns:
        np.ndarray | None: RGB 画像．入力が無い場合は None
    """
    if image is None:
        return None
    if isinstance(image, Image.Image):
        return np.asarray(image.convert("RGB"))
    arr = np.asarray(image)
    if arr.ndim == 2:
        return cv2.cvtColor(arr.astype(np.uint8), cv2.COLOR_GRAY2RGB)
    if arr.shape[2] == 4:
        return arr[:, :, :3].astype(np.uint8)
    return arr.astype(np.uint8)


def detect_largest_face(
    rgb: np.ndarray,
    cascade: cv2.CascadeClassifier,
    margin: float = 0.25,
) -> tuple[int, int, int, int] | None:
    """画像から最も大きい顔のバウンディングボックスを返す．

    Args:
        rgb (np.ndarray): RGB 画像，形状 (H, W, 3)
        cascade (cv2.CascadeClassifier): 顔検出器
        margin (float): 検出枠に対する余白倍率

    Returns:
        tuple[int, int, int, int] | None: (x1, y1, x2, y2)．未検出時は None
    """
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    faces = cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(48, 48),
    )
    if len(faces) == 0:
        return None
    x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
    pad_x = int(w * margin)
    pad_y = int(h * margin)
    h_img, w_img = rgb.shape[:2]
    x1 = max(0, x - pad_x)
    y1 = max(0, y - pad_y)
    x2 = min(w_img, x + w + pad_x)
    y2 = min(h_img, y + h + pad_y)
    return x1, y1, x2, y2


def draw_face_box(
    rgb: np.ndarray,
    box: tuple[int, int, int, int],
    label: str,
) -> np.ndarray:
    """顔枠とラベルを描画した画像を返す．

    Args:
        rgb (np.ndarray): RGB 画像，形状 (H, W, 3)
        box (tuple[int, int, int, int]): (x1, y1, x2, y2)
        label (str): 枠の上に書く文字列

    Returns:
        np.ndarray: 描画後の RGB 画像，形状 (H, W, 3)
    """
    vis = rgb.copy()
    x1, y1, x2, y2 = box
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 180, 80), 3)

    # 日本語ラベルは PIL で描画（OpenCV の putText は日本語非対応）
    pil = Image.fromarray(vis)
    draw = ImageDraw.Draw(pil)
    try:
        font = ImageFont.truetype(str(FONT_PATH), 28)
    except OSError:
        font = ImageFont.load_default()
    text_y = max(0, y1 - 36)
    draw.text((x1 + 4, text_y), label, fill=(0, 180, 80), font=font)
    return np.asarray(pil)


def format_predictions(preds: list[dict]) -> str:
    """分類結果を日本語の読みやすいテキストにする．

    Args:
        preds (list[dict]): pipeline の出力（label / score）

    Returns:
        str: ランキング形式の文字列
    """
    lines: list[str] = []
    for i, item in enumerate(preds, start=1):
        en = str(item["label"]).lower()
        ja = EMOTION_JA.get(en, en)
        score = float(item["score"]) * 100.0
        bar = "█" * int(round(score / 5.0))
        lines.append(f"{i}. {ja}（{en}）: {score:5.1f}%  {bar}")
    top = preds[0]
    top_en = str(top["label"]).lower()
    top_ja = EMOTION_JA.get(top_en, top_en)
    header = f"【推定結果】{top_ja}（確信度 {float(top['score']) * 100:.1f}%）\n\n"
    return header + "\n".join(lines)


def predict_emotion(image, mirror: bool = False) -> tuple[np.ndarray | None, str]:
    """顔写真から表情を推定する（Gradio コールバック）．

    Args:
        image: Gradio Image 入力（カメラ／アップロード／サンプル）
        mirror (bool): True のとき左右反転してから処理する

    Returns:
        tuple[np.ndarray | None, str]: (可視化画像, 結果テキスト)
    """
    rgb = to_rgb_uint8(image)
    if rgb is None:
        return None, "画像がありません．カメラ撮影・アップロード・サンプルのいずれかを選んでください．"

    if mirror:
        rgb = np.ascontiguousarray(rgb[:, ::-1, :])

    with tqdm(total=2, desc="表情認識", leave=False) as pbar:
        box = detect_largest_face(rgb, face_cascade, margin=FACE_MARGIN)
        pbar.update(1)
        if box is None:
            pbar.update(1)
            msg = (
                "顔を検出できませんでした．\n"
                "顔が正面・明るく・大きく写るようにして，もう一度お試しください．"
            )
            return rgb, msg

        x1, y1, x2, y2 = box
        face_rgb = rgb[y1:y2, x1:x2]
        face_pil = Image.fromarray(face_rgb)
        preds = emotion_clf(face_pil, top_k=7)
        pbar.update(1)

    # pipeline は top_k=1 のとき dict を返す場合がある
    if isinstance(preds, dict):
        preds = [preds]

    top_en = str(preds[0]["label"]).lower()
    top_ja = EMOTION_JA.get(top_en, top_en)
    score = float(preds[0]["score"]) * 100.0
    vis = draw_face_box(rgb, box, f"{top_ja} {score:.0f}%")
    text = format_predictions(preds)
    return vis, text


def build_demo(
    sample_items: list[tuple[str, Path]],
    mirror_webcam: bool,
) -> gr.Blocks:
    """Gradio UI を構築する．

    Args:
        sample_items (list[tuple[str, Path]]): (表示ラベル, 画像パス)
        mirror_webcam (bool): カメラ入力を左右反転するか

    Returns:
        gr.Blocks: Gradio デモ
    """
    example_paths = [str(path) for _, path in sample_items]

    with gr.Blocks(title="表情認識デモ") as demo:
        gr.Markdown(
            """
            # 表情認識デモ
            顔写真から **7種類の表情**（喜び・悲しみ・怒り・驚き・恐れ・嫌悪・無表情）を推定します．  
            **カメラ**で撮影するか，下の**サンプル画像**をクリックして試せます．
            """
        )
        with gr.Row():
            with gr.Column():
                image_in = gr.Image(
                    label="顔写真（カメラ / アップロード）",
                    type="numpy",
                    sources=["webcam", "upload"],
                    mirror_webcam=mirror_webcam,
                )
                mirror_flag = gr.Checkbox(
                    label="入力画像を左右反転して認識する",
                    value=False,
                    info="アップロード画像の向きが逆のときだけオンにしてください（カメラは上のミラー設定を利用）",
                )
                run_btn = gr.Button("表情を認識", variant="primary")
            with gr.Column():
                image_out = gr.Image(label="検出結果（顔枠）", type="numpy")
                text_out = gr.Textbox(label="推定結果", lines=12)

        gr.Examples(
            examples=example_paths,
            inputs=[image_in],
            label="サンプル画像（クリックで入力）",
            examples_per_page=8,
        )

        run_btn.click(
            fn=predict_emotion,
            inputs=[image_in, mirror_flag],
            outputs=[image_out, text_out],
        )
        image_in.change(
            fn=predict_emotion,
            inputs=[image_in, mirror_flag],
            outputs=[image_out, text_out],
        )
    return demo


# ------------------------------------------------------------
# 初期化
# ------------------------------------------------------------
device = resolve_device()
download_font(FONT_URL, FONT_PATH)
print(f"フォント: {FONT_PATH} ({FONT_PATH.stat().st_size} bytes)")
sample_items = prepare_sample_images(SAMPLE_IMAGE_SOURCES, SAMPLE_DIR)
face_cascade = load_face_cascade()
emotion_clf = load_emotion_pipeline(MODEL_ID, device)
print("初期化完了．次のセルで Gradio を起動してください．")


## 3. Gradio の実行

UI が起動したら，サンプル画像をクリックするか，カメラで顔を撮影して「表情を認識」を押してください．


In [ ]:
demo = build_demo(sample_items, mirror_webcam=MIRROR_WEBCAM)
demo.launch(share=True, debug=False)
